# Pricer d'Autocall en Monte Carlo

## Produit structuré sur une action du CAC 40

Ce projet implémente le pricing d'un **autocall** (produit structuré à barrières, remboursement anticipé conditionnel et coupon conditionnel) par simulation Monte Carlo, avec calcul des sensibilités (Greeks) par différences finies.

Contrairement à une option vanille qui ne dépend que du prix final du sous-jacent, un autocall dépend de **toute la trajectoire** du prix (est-ce qu'il a touché la barrière de rappel à chaque date d'observation ? est-ce qu'il a touché la barrière de protection à un moment donné ?). C'est pour ça qu'on ne peut pas utiliser une formule fermée comme Black-Scholes : il faut simuler un grand nombre de trajectoires possibles et faire la moyenne des résultats.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Caractéristiques du produit

On modélise un autocall "Athena" classique (la structure la plus courante), sur une action réelle — TotalEnergies, pour rester dans la continuité du projet momentum, et parce qu'on a déjà sa volatilité réelle (~28,6% annualisée, calculée dans le projet précédent).

**Mécanique du produit :**
- Observation chaque année pendant 3 ans (3 dates : 1 an, 2 ans, 3 ans)
- À chaque date d'observation, si le sous-jacent est **au-dessus de la barrière de rappel** (100% du niveau initial) : le produit est remboursé automatiquement, l'investisseur reçoit son capital **+ un coupon cumulé** (par année écoulée) — et le produit s'arrête là
- Si on arrive à l'échéance (3 ans) sans jamais avoir été rappelé :
  - si le sous-jacent est au-dessus de la **barrière de protection** (60% du niveau initial) : l'investisseur récupère son capital, sans coupon
  - si le sous-jacent est en-dessous de cette barrière : l'investisseur subit la baisse du sous-jacent à 100% (perte en capital)

C'est ce dernier cas qui rend le produit risqué : contrairement à une obligation, le capital n'est protégé que si le marché ne s'effondre pas.

In [ ]:
# Paramètres de marché
S0 = 100            # niveau initial du sous-jacent, normalisé à 100 (= 100% de la valeur de départ)
r = 0.03             # taux sans risque (3%, cohérent avec les hypothèses des projets précédents)
sigma = 0.286        # volatilité annualisée réelle de TotalEnergies, calculée dans le projet momentum
q = 0.03             # taux de dividende du sous-jacent (3%/an) — important : plus le dividende est élevé,
                     # plus le coupon que la banque peut offrir est élevé (voir section suivante)

# Caractéristiques du produit
T = 3                                    # maturité : 3 ans
observation_dates = np.array([1, 2, 3])  # dates d'observation annuelles (en années depuis l'émission)
coupon_rate = 0.08                        # coupon de 8% par année écoulée depuis l'émission, versé si rappel (point de départ arbitraire)
autocall_barrier = 1.00 * S0              # barrière de rappel anticipé : 100% du niveau initial
protection_barrier = 0.60 * S0            # barrière de protection du capital : 60% du niveau initial
notional = 100                            # montant investi (base 100, pour raisonner en %)

print(f"Barrière de rappel      : {autocall_barrier} (100% de S0)")
print(f"Barrière de protection  : {protection_barrier} (60% de S0)")
print(f"Coupon annuel si rappelé : {coupon_rate:.0%}")

## 2. Simulation des trajectoires du sous-jacent

On simule le prix du sous-jacent sous un **mouvement brownien géométrique** (le même modèle que pour Black-Scholes) :

S(t) = S(t-1) × exp[(r − q − σ²/2) × dt + σ × √dt × Z]

où Z est un tirage aléatoire suivant une loi normale standard. C'est la même logique que la simulation Monte Carlo du pricer d'options, mais avec une différence importante : comme le produit n'observe le sous-jacent qu'aux **dates d'observation** (1 an, 2 ans, 3 ans) et non en continu, on n'a besoin de simuler le prix qu'à ces 3 instants précis, pas jour par jour. C'est une simplification réaliste (les autocalls réels fonctionnent avec des dates d'observation discrètes, pas une barrière surveillée en continu) qui rend la simulation beaucoup plus rapide.

Le terme `r − q` (taux sans risque moins dividende) est important : plus le dividende est élevé, plus la dérive du sous-jacent sous la mesure risque-neutre est faible — c'est ce qui permet à la banque d'offrir un coupon attractif (un sous-jacent qui verse de gros dividendes "monte" moins vite en moyenne sous cette mesure, ce qui rend le produit plus difficile à rappeler, donc la banque peut compenser avec un coupon plus élevé).

In [ ]:
np.random.seed(42)                                   # pour des résultats reproductibles
n_sims = 100_000                                      # nombre de trajectoires simulées

dt = np.diff(np.concatenate([[0], observation_dates])) # durée entre chaque date d'observation : [1, 1, 1] (en années)

Z = np.random.standard_normal((n_sims, len(observation_dates)))  # tirages aléatoires indépendants (n_sims x 3 dates)

log_returns = (r - q - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z  # rendement log GBM sur chaque intervalle
paths = S0 * np.exp(np.cumsum(log_returns, axis=1))                    # prix cumulé à t=1, t=2, t=3 pour chaque simulation

paths[:5]                                              # aperçu : 5 trajectoires, colonnes = [S(1), S(2), S(3)]

## 3. Fonction de payoff

C'est l'étape la plus importante : traduire en code la mécanique du produit décrite en partie 1, pour chacune des 100 000 trajectoires simulées.

On traite les 3 dates d'observation **dans l'ordre chronologique**, et on garde la trace des trajectoires déjà "rappelées" (`redeemed`) pour ne jamais les retraiter aux dates suivantes — une fois remboursée, une trajectoire sort du jeu.

- À chaque date, si une trajectoire pas encore rappelée est au-dessus de la barrière de rappel, elle reçoit son capital + le coupon cumulé (coupon × nombre d'années écoulées), et sort du jeu.
- Pour toutes les trajectoires qui arrivent à l'échéance sans avoir été rappelées : si elles sont au-dessus de la barrière de protection, elles récupèrent juste leur capital ; sinon elles subissent la baisse du sous-jacent (perte proportionnelle).

On actualise chaque flux à sa propre date de versement avec `exp(-r × t)` : un remboursement à 1 an n'a pas la même valeur aujourd'hui qu'un remboursement à 3 ans, même montant.

In [ ]:
payoffs = np.zeros(n_sims)              # le montant versé à l'investisseur, pour chaque simulation
discount_factors = np.zeros(n_sims)      # le facteur d'actualisation correspondant à la date de ce versement
redeemed = np.zeros(n_sims, dtype=bool)  # True si la trajectoire a déjà été rappelée par anticipation

for i, t in enumerate(observation_dates):
    S_t = paths[:, i]                                          # niveau du sous-jacent à cette date d'observation
    newly_called = (~redeemed) & (S_t >= autocall_barrier)      # pas encore rappelées, et condition de rappel remplie aujourd'hui
    payoffs[newly_called] = notional * (1 + coupon_rate * t)    # capital + coupon cumulé depuis l'émission
    discount_factors[newly_called] = np.exp(-r * t)
    redeemed |= newly_called                                    # on les marque comme rappelées, pour les ignorer ensuite

# Trajectoires jamais rappelées : on regarde le niveau final, à l'échéance
not_redeemed = ~redeemed
S_final = paths[:, -1]

above_protection = not_redeemed & (S_final >= protection_barrier)
payoffs[above_protection] = notional                            # capital rendu intégralement, mais sans coupon
discount_factors[above_protection] = np.exp(-r * T)

below_protection = not_redeemed & (S_final < protection_barrier)
payoffs[below_protection] = notional * (S_final[below_protection] / S0)  # perte en capital, proportionnelle à la baisse
discount_factors[below_protection] = np.exp(-r * T)

discounted_payoffs = payoffs * discount_factors                 # valeur actualisée de chaque flux, aujourd'hui

price = discounted_payoffs.mean()                                # le prix Monte Carlo = moyenne des flux actualisés
print(f"Prix de l'autocall (pour 100 investis) : {price:.2f}")

## 4. Calibration du coupon (le vrai travail du structureur)

Plutôt que de fixer le coupon au hasard, on cherche la valeur de `coupon_rate` qui rend le prix du produit exactement égal à 100 (le pair — ce que l'investisseur paie réellement). C'est le sens inverse du pricing classique : on connaît le prix cible, on cherche le paramètre du produit qui l'atteint — exactement la démarche d'un desk de structuring, qui part des conditions de vente voulues et calibre le produit en conséquence.

On utilise une recherche par dichotomie (bissection) : comme le prix est une fonction croissante du coupon (plus le coupon est élevé, plus le produit vaut cher), on peut encadrer la solution et resserrer l'intervalle à chaque itération.

In [ ]:
def price_autocall(coupon_rate, paths):                          # on regroupe le calcul dans une fonction,
    payoffs = np.zeros(n_sims)                                    # pour pouvoir l'appeler avec différents coupons sans tout réécrire
    discount_factors = np.zeros(n_sims)
    redeemed = np.zeros(n_sims, dtype=bool)

    for i, t in enumerate(observation_dates):
        S_t = paths[:, i]
        newly_called = (~redeemed) & (S_t >= autocall_barrier)
        payoffs[newly_called] = notional * (1 + coupon_rate * t)
        discount_factors[newly_called] = np.exp(-r * t)
        redeemed |= newly_called

    not_redeemed = ~redeemed
    S_final = paths[:, -1]
    above = not_redeemed & (S_final >= protection_barrier)
    payoffs[above] = notional
    discount_factors[above] = np.exp(-r * T)
    below = not_redeemed & (S_final < protection_barrier)
    payoffs[below] = notional * (S_final[below] / S0)
    discount_factors[below] = np.exp(-r * T)

    return (payoffs * discount_factors).mean()

low, high = 0.0, 0.30                 # on encadre la solution entre 0% et 30% de coupon annuel
target_price = 100.0

for _ in range(40):                   # 40 itérations suffisent largement pour converger à une précision négligeable
    mid = (low + high) / 2
    p = price_autocall(mid, paths)
    if p < target_price:
        low = mid                     # le prix est trop bas : il faut un coupon plus généreux
    else:
        high = mid                    # le prix est trop haut : le coupon actuel est déjà suffisant

fair_coupon = (low + high) / 2
print(f"Coupon juste (prix = 100) : {fair_coupon:.2%}")
print(f"Prix avec ce coupon        : {price_autocall(fair_coupon, paths):.2f}")

**Pourquoi 18,75% ?** Ce chiffre peut surprendre, mais il s'explique : la barrière de rappel est fixée à 100% du niveau initial (assez difficile à dépasser avec un sous-jacent volatil à 28,6%), et surtout il n'y a aucun plafond ni filet en dessous de 60% — si le marché s'effondre, l'investisseur perd à 100% la baisse du sous-jacent. Pour compenser ce risque de queue, un coupon beaucoup plus généreux est nécessaire pour que le produit vaille 100 en moyenne. Les vrais autocalls du marché ont souvent une barrière de rappel qui **descend avec le temps**, justement pour limiter ce besoin de coupon extrême (voir les limites, en fin de notebook).

## 5. Répartition des scénarios

Le prix est une moyenne, mais elle cache des trajectoires très différentes : certaines sont rappelées dès la première année, d'autres encaissent une perte en capital. Pour un produit vendu à des clients, c'est souvent ce tableau de probabilités — plus que le prix lui-même — qui est présenté commercialement : "quelle chance ai-je d'être remboursé rapidement ? De perdre du capital ?"


In [ ]:
payoffs = np.zeros(n_sims)
discount_factors = np.zeros(n_sims)
redeemed = np.zeros(n_sims, dtype=bool)
redeemed_at = np.zeros(n_sims, dtype=int)        # 0 = jamais rappelé, sinon le numéro de la date (1, 2 ou 3)

for i, t in enumerate(observation_dates):
    S_t = paths[:, i]
    newly_called = (~redeemed) & (S_t >= autocall_barrier)
    payoffs[newly_called] = notional * (1 + fair_coupon * t)
    discount_factors[newly_called] = np.exp(-r * t)
    redeemed_at[newly_called] = i + 1
    redeemed |= newly_called

not_redeemed = ~redeemed
S_final = paths[:, -1]
above_protection = not_redeemed & (S_final >= protection_barrier)
below_protection = not_redeemed & (S_final < protection_barrier)

payoffs[above_protection] = notional
discount_factors[above_protection] = np.exp(-r * T)
payoffs[below_protection] = notional * (S_final[below_protection] / S0)
discount_factors[below_protection] = np.exp(-r * T)

scenarios = pd.Series({
    "Rappelé en année 1": (redeemed_at == 1).mean(),
    "Rappelé en année 2": (redeemed_at == 2).mean(),
    "Rappelé en année 3": (redeemed_at == 3).mean(),
    "Jamais rappelé, capital protégé (sans coupon)": above_protection.mean(),
    "Jamais rappelé, perte en capital": below_protection.mean(),
})

print(scenarios.apply(lambda x: f"{x:.1%}"))
print(f"\nPerte moyenne (sur les seuls scénarios de perte) : {(1 - S_final[below_protection]/S0).mean():.1%}")

**Lecture** : 61,2% de chances d'être rappelé avec un gain (année 1, 2 ou 3 confondues), mais quand ça tourne mal (18,4% des cas), la perte moyenne est sévère (-54,4%). C'est un profil "gain probable mais limité, perte rare mais grosse" — très différent d'un placement classique, et c'est exactement ce qu'un vendeur de produits structurés doit savoir expliquer clairement à un client, sans enjoliver.

## 6. Visualisation des scénarios

Un graphique en barres pour la répartition des issues, et un histogramme des rendements obtenus par l'investisseur (par rapport aux 100 investis), pour voir d'un coup d'œil la forme du risque : beaucoup de petits gains groupés, une queue de pertes étalée à gauche.

In [ ]:
investor_return = payoffs / notional - 1   # rendement de l'investisseur (non actualisé), en % du capital investi

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scenarios.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Répartition des scénarios")
axes[0].set_xlabel("Probabilité")

axes[1].hist(investor_return, bins=80, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_title("Distribution du rendement final de l'investisseur")
axes[1].set_xlabel("Rendement (non actualisé)")

plt.tight_layout()
plt.savefig("scenarios_autocall.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Sensibilités (Greeks) par différences finies

Contrairement à Black-Scholes, il n'existe pas de formule fermée pour les Greeks d'un autocall — on les estime donc par **différences finies** ("bump-and-reval") : on reprice le produit avec un paramètre légèrement modifié, et on observe de combien le prix a bougé.

- **Delta** : sensibilité au niveau du sous-jacent. On reprice avec S0 +1%, puis S0 -1%, et on mesure la pente — c'est l'équivalent, pour ce produit complexe, du delta d'une option vanille.
- **Vega** : sensibilité à la volatilité. On reprice avec σ +1 point de volatilité, pour voir si le produit vaut plus ou moins cher avec plus d'incertitude sur le marché.

Fait notable à anticiper : pour un autocall, le **vega est souvent négatif** (contrairement à une option vanille, toujours vega positif) — une volatilité plus élevée augmente le risque de franchir la barrière de protection à la baisse, ce qui peut peser plus lourd que l'avantage d'un rappel plus probable. On va vérifier si c'est le cas ici.

In [ ]:
def simulate_paths(S0_, sigma_, seed=42):                          # on isole la simulation dans une fonction,
    np.random.seed(seed)                                            # pour pouvoir la relancer avec un paramètre modifié
    Z = np.random.standard_normal((n_sims, len(observation_dates)))
    log_returns = (r - q - 0.5 * sigma_**2) * dt + sigma_ * np.sqrt(dt) * Z
    return S0_ * np.exp(np.cumsum(log_returns, axis=1))

# --- Delta : sensibilité à S0 ---
bump_S = 0.01 * S0                                                  # on bouge le spot de 1%
paths_up = simulate_paths(S0 + bump_S, sigma)
paths_down = simulate_paths(S0 - bump_S, sigma)

price_up = price_autocall(fair_coupon, paths_up)
price_down = price_autocall(fair_coupon, paths_down)
delta = (price_up - price_down) / (2 * bump_S)

print(f"Delta : {delta:.4f}  (le prix bouge de {delta:.2f} pour 1 point de variation du sous-jacent)")

# --- Vega : sensibilité à sigma ---
bump_vol = 0.01                                                     # +1 point de volatilité (28.6% -> 29.6%)
paths_vol_up = simulate_paths(S0, sigma + bump_vol)
price_vol_up = price_autocall(fair_coupon, paths_vol_up)

price_base = price_autocall(fair_coupon, paths)
vega = price_vol_up - price_base                                    # directement la variation de prix pour ce +1 point de vol
                                                                      # (pas de division par bump_vol : bump_vol EST l'unité qu'on veut)

print(f"Vega  : {vega:.4f}  (le prix bouge de {vega:.2f} pour +1 point de volatilité)")

**Lecture** : le vega est bien négatif (-0,68) comme anticipé. Plus de volatilité augmente surtout le risque de passer sous la barrière de protection à 60% (une queue de perte qui s'aggrave), et cet effet pèse plus lourd que l'avantage d'un rappel légèrement plus probable. C'est une information cruciale pour un sales ou un structureur : **vendre un autocall, c'est vendre de la volatilité** (la banque qui l'émet est "short vega"), ce qui explique pourquoi ces produits sont particulièrement rentables à structurer en période de vol élevée.

## Conclusion

| | Valeur |
|---|---|
| Prix avec coupon à 8% (arbitraire) | 91,44 (pour 100 investis) |
| **Coupon juste (prix = 100)** | **18,75% par an** |
| Probabilité de rappel avec gain (an 1, 2 ou 3) | 61,2% |
| Probabilité de capital rendu sans coupon | 20,4% |
| Probabilité de perte en capital | 18,4% (perte moyenne : -54,4%) |
| Delta | 0,54 |
| Vega | -0,68 |

**Résultat** : la mécanique de pricing fonctionne — on simule, on actualise, et surtout on calibre le coupon comme le ferait un vrai structureur (plutôt que de le fixer au hasard). Le profil de risque est clairement asymétrique : une majorité de scénarios favorables mais limités en gain, contre une minorité de scénarios de perte sévère — le profil "vendeur de volatilité" typique des produits à barrière. Le vega négatif confirme ce point.

**Limites et pistes d'amélioration** :
- Le modèle utilise une volatilité **constante** (mouvement brownien géométrique classique) ; en réalité, les desks de structuring pricent à partir d'une **surface de volatilité** (vol implicite qui varie selon le strike et la maturité), ce qui changerait sensiblement le vega et le coupon juste
- Les observations sont annuelles ; beaucoup d'autocalls réels observent trimestriellement, ce qui change la probabilité de rappel anticipé
- Le **risque de crédit de l'émetteur** n'est pas modélisé : un autocall est une dette de la banque qui l'émet, donc l'investisseur porte aussi le risque que la banque fasse défaut — ce risque est généralement intégré au pricing réel via le taux d'actualisation
- La barrière de rappel est fixe à 100% ; les produits réels ont souvent une barrière **dégressive** (plus basse chaque année), ce qui change fortement le coupon nécessaire (observé ici : 18,75% est élevé précisément à cause de cette barrière fixe difficile à atteindre)
- Les taux et dividendes sont supposés constants sur toute la durée, alors qu'ils varient dans la réalité

In [ ]:
pd.Series({
    "Prix (coupon 8%)": 91.44,
    "Coupon juste": fair_coupon,
    "Delta": delta,
    "Vega": vega,
}).to_csv("resultats_autocall.csv")

scenarios.to_csv("scenarios_autocall.csv")